# 🛡️ Detector UEBA: Detecção de Anomalias em Logs de Autenticação
### Dissertação de Mestrado Profissional em Engenharia Elétrica (UnB / FT / ENE)
* **Autor:** Allysson de Oliveira Noronha
* **Orientador:** Prof. Dr. Robson de Oliveira Albuquerque
* **Repositório:** [https://github.com/allyssonnor/anti_Detector](https://github.com/allyssonnor/anti_Detector)

---

### 🎯 Propósito deste Demonstrador
Este notebook foi elaborado para que a **Banca Examinadora** possa reproduzir e auditar em minutos o pipeline experimental:
1. **Auditoria de Causalidade (`pytest`):** Garantia matemática e algorítmica de ausência de vazamento temporal (*data leakage*).
2. **Treinamento e Calibração Dinâmica de Limiares:** Treino cronológico no dataset de referência EvidenceForge com otimização dinâmica de limiares operacionais via $F_2$-Score na validação para ambos os modelos: **Random Forest** (supervisionado) e **MLP Autoencoder** (reconstrutivo $X \to X$).
3. **Inferência Cruzada e Generalização:** Aplicação do modelo treinado sobre datasets de teste independentes sem recalibração (*zero-shot domain transfer*).
4. **Dashboard Forense e Métricas:** Visualização imediata de Matriz de Confusão, PR-AUC, ROC-AUC, Lift e explicabilidade dos eventos detectados.

---

### 📦 Datasets Disponíveis no Repositório (`data/`)
1. **`data/ef_amplo.jsonl`:** Dataset multimodal EvidenceForge (20.355 eventos) contendo atividade legítima de expediente corporativo e 3 ataques canônicos (Força Bruta, Acesso Fora de Hora e Movimentação Lateral via RDP).
2. **`data/ef_bruteforce.jsonl`:** Dataset especialista EvidenceForge (19.032 eventos) com expediente corporativo legítimo e exclusivamente campanhas de Força Bruta e Password Spraying.
3. **`data/test_bruteforce_42.jsonl`:** Dataset de teste independente para avaliação confirmatória de transferência de domínio contra ataques de força bruta (122.239 eventos).
4. **`data/seed_1_sample.jsonl`:** Amostra representativa do dataset massivo para verificação de escala e robustez.

## 1. ⚙️ Preparação do Ambiente e Clonagem do Repositório

In [ ]:
# 1. Clonagem resiliente do repositório (diretório fixo /content/anti_Detector)
import os

%cd /content
if not os.path.isdir("/content/anti_Detector"):
    !git clone https://github.com/allyssonnor/anti_Detector.git
%cd /content/anti_Detector
!git pull

# 2. Instalação das dependências necessárias
!pip install -q -r requirements.txt


## 2. 🧪 Auditoria de Integridade e Causalidade Temporal
Execução da suíte de testes de causalidade para comprovar que nenhuma feature utiliza informação futura no instante $t$.

In [ ]:
!python -m pytest tests/test_causality.py -v

## 3. 🏋️ Treinamento e Calibração Dinâmica de Limiares
Execução do protocolo de treino anti-leakage com particionamento cronológico (70% treino, 20% validação, 10% teste misto).
O pipeline treina e calibra os limiares de ambos os modelos:
* **RandomForest** (classificador comitê supervisionado)
* **MLP_Single** (Autoencoder neural baseado em erro de auto-reconstrução $X \to X$)


In [ ]:
!python scripts/detectar.py --exp_config config/demo_colab.yaml

## 4. 📊 Tabela Consolidada de Resultados de Validação

In [ ]:
import pandas as pd
import glob

# Busca resiliente pelo consolidado master_results_table.csv
res_csv = glob.glob("**/master_results_table.csv", recursive=True)
if not res_csv:
    res_csv = glob.glob("results/**/master_results*.csv", recursive=True)

if res_csv:
    df_res = pd.read_csv(res_csv[0])
    cols = [c for c in ["model_preset", "dataset", "precision", "recall", "f1_score", "f2_score", "pr_auc", "roc_auc", "lift_pr", "tp", "fp", "fn", "tn"] if c in df_res.columns]
    print("\n📈 TABELA CONSOLIDADA DE RESULTADOS (Modelos Avaliados):")
    display(df_res[cols].drop_duplicates())
else:
    print("Nenhum resultado consolidado encontrado. Verifique se o treino na Célula 3 concluiu.")


## 5. 🔍 Inferência Cruzada em Cenário Independente (Zero-Shot Transfer)
Aplicação do modelo treinado e dos limiares calibrados em dados nunca vistos:
* `data/test_bruteforce_42.jsonl` (Ataque isolado de força bruta / password guessing)
* `data/seed_1_sample.jsonl` (Amostra multimodal do dataset massivo)
* `data/ef_bruteforce.jsonl` (Cenário especialista EvidenceForge)

In [ ]:
# Escolha do modelo e do dataset de teste para inferência cruzada:
MODEL_DIR = "results/demo_colab/full/run_seed42/Synthetic_Chronological_RandomForest_global_20"
DATA_TARGET = "data/test_bruteforce_42.jsonl"  # Alterne para data/seed_1_sample.jsonl ou data/ef_bruteforce.jsonl

!python scripts/inferir_usual.py --config config/demo_colab.yaml --data $DATA_TARGET --model_dir $MODEL_DIR


## 6. 🎨 Dashboard Forense e Explicabilidade dos Alertas

In [ ]:
from IPython.display import Image, display
import glob
import os
import pandas as pd

# Busca pelo dashboard gerado na inferência mais recente
dashboards = glob.glob(f"{MODEL_DIR}/dashboard_*.png")
if dashboards:
    display(Image(dashboards[0]))
else:
    print(f"Dashboard não encontrado em: {MODEL_DIR}")

# Amostra das detecções e anomalias identificadas
eventos_csvs = glob.glob(f"{MODEL_DIR}/inferencia_*.csv")
if eventos_csvs:
    df_inf = pd.read_csv(eventos_csvs[0])
    print("\n🛡️ Top 10 Alertas Disparados com Detalhes Comportamentais:")
    alertas = df_inf[df_inf["alert"] == 1] if "alert" in df_inf.columns else df_inf.head(10)
    cols_show = [c for c in ["UserID", "Time", "IpAddress", "LogHost", "anomaly_score", "threshold", "alert", "Is_Anomaly", "AnomalyType"] if c in df_inf.columns]
    display(alertas[cols_show].head(10))
